# 🏎️ F1 Strategy Predictor - LSTM Model
## Predizione Pit Stop e Compound con Deep Learning

---

### Obiettivo del Progetto
Questo notebook risponde alle seguenti domande:

1. **È possibile prevedere quando un pilota effettuerà un pit stop?**
   - Predizione binaria: pit nei prossimi 3 giri? (Sì/No)
   - Metrica target: AUC-ROC ≥ 0.90

2. **È possibile prevedere quale compound verrà montato al prossimo pit?**
   - Predizione multiclass: SOFT, MEDIUM, HARD, INTERMEDIATE, WET
   - Metrica target: Accuracy ≥ 0.75


### Approccio
Due modelli LSTM separati con Attention mechanism:
- **Modello PIT**: Classificazione binaria con focus su degrado gomme e timing
- **Modello COMPOUND**: Classificazione multiclass con focus su strategia e condizioni

### Prerequisiti
- `f1_dataset_clean.pkl` (generato dal notebook DataAnalysis)

# 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os, sys
import json

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.model_selection import train_test_split


pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [2]:
import joblib
import json
import os

os.makedirs('Model', exist_ok=True)
os.makedirs('Other', exist_ok=True)

In [3]:
# FastF1: libreria open-source per dati F1 (telemetria, tempi, meteo)
import importlib.util
if importlib.util.find_spec('fastf1') is None:
    !pip install fastf1 --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.0/123.0 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-server 2.14.0 requires websocket-client>=1.7, but you have websocket-client 1.0.0 which is incompatible.


# 2. Import Dataset

In [4]:
# Carica il dataset già pulito (da notebook DataAnalysis)
# Le seguenti operazioni sono già state effettuate:
#   - Conversione Timedelta → secondi (*Sec)
#   - Unificazione nomi team
#   - Rimozione compound non validi

df_f1 = pd.read_pickle('f1_dataset_clean.pkl')

print(f"Dataset caricato: {len(df_f1):,} righe")
print(f"\nTeam: {list(df_f1['Team'].unique())}")
print(f"\nCompound: {list(df_f1['Compound'].unique())}")

Dataset caricato: 86,757 righe

Team: ['Ferrari', 'Red Bull Racing', 'Mercedes', 'Haas F1 Team', 'Racing Bulls', 'Alpine', 'Williams', 'Kick Sauber', 'Aston Martin', 'McLaren']

Compound: ['SOFT', 'MEDIUM', 'HARD', 'INTERMEDIATE', 'WET']


# 3. Feature Engineering

---
## Targets

In [5]:
# Ordinamento cronologico: essenziale per sequenze temporali!
df_f1 = df_f1.sort_values(['Year', 'Round', 'Driver', 'LapNumber']).reset_index(drop=True)

**TARGET 1: PitIn3Laps** (Binary)
- 1 se pit entro i prossimi 3 giri, 0 altrimenti
- *Perché 3 giri?* Una finestra più ampia cattura l'imminenza del pit senza essere troppo generica. Un pit "imminente" a 1 giro è troppo restrittivo (poche sample positive).

In [6]:
# --- TARGET 1: PitIn3Laps ---

# Identifica i giri in cui il pilota pittarà entro i prossimi 3
df_f1['NextLapStint'] = df_f1.groupby(['Year', 'Round', 'Driver'])['Stint'].shift(-1)
# Lo shift -1 consente di spostare la colonna Stint verso l'alto = giro seguente
df_f1['IsPitLap'] = (df_f1['NextLapStint'] != df_f1['Stint']).astype(float).fillna(0)
# Fillna(0) sostituisce valori mancanti con 0. IsPtLap avrà valore 0/1.

# Aggregazione su finestra di 3 giri
df_f1['PitIn1'] = df_f1.groupby(['Year', 'Round', 'Driver'])['IsPitLap'].shift(-1).fillna(0)
# Lo shift -1 consente di spostare la colonna Stint verso l'alto = giro seguente
# Fillna(0) sostituisce valori NaN con 0 (ex. se considero ultimo giro)
df_f1['PitIn2'] = df_f1.groupby(['Year', 'Round', 'Driver'])['IsPitLap'].shift(-2).fillna(0)
df_f1['PitIn3'] = df_f1.groupby(['Year', 'Round', 'Driver'])['IsPitLap'].shift(-3).fillna(0)

df_f1['PitIn3Laps'] = ((df_f1['IsPitLap'] + df_f1['PitIn1'] + df_f1['PitIn2']
                        + df_f1['PitIn3']) > 0).astype(int)

In [7]:
# Pulizia colonne temporanee
df_f1 = df_f1.drop(columns=['PitIn1', 'PitIn2', 'PitIn3'], errors='ignore')

print(f"Target PitIn3Laps: {df_f1['PitIn3Laps'].mean():.1%} giri con pit previsto entro 3 laps ({df_f1['PitIn3Laps'].sum():,} su {len(df_f1):,})")

Target PitIn3Laps: 20.7% giri con pit previsto entro 3 laps (18,001 su 86,757)


**TARGET 2: NextCompound** (Multiclass)
- Il compound (tipo di gomma) che verrà montato al prossimo pit stop

In [8]:
# Per ogni giro, trova il compound del prossimo stint.
def get_next_compound(group):
    group = group.sort_values('LapNumber').copy()
    stints = group['Stint'].values
    compounds = group['StintCompound'].values
    result = compounds.copy()

    for i in range(len(group)):
        for j in range(i + 1, len(group)):
            if stints[j] != stints[i]:
                result[i] = compounds[j]
                break
    return result

In [9]:
# --- TARGET 2: NextCompound ---

# Per ogni gruppo, estrae il primo valore della colonna Compound,
# che rappresenta il tipo di gomma montata nel primo giro dello stint.
df_f1['StintCompound'] = df_f1.groupby(['Year', 'Round', 'Driver', 'Stint'])['Compound'].transform('first')

next_compounds = []
for name, group in df_f1.groupby(['Year', 'Round', 'Driver']):
    next_compounds.extend(get_next_compound(group))
df_f1['NextCompound'] = next_compounds

In [10]:
# Pulizia colonne temporanee
df_f1 = df_f1.drop(columns=['NextLapStint', 'StintCompound'], errors='ignore')

print(df_f1['NextCompound'].value_counts())

NextCompound
HARD            45065
MEDIUM          24448
SOFT            13143
INTERMEDIATE     3971
WET               130
Name: count, dtype: int64


## Features Aggregate

Le features sono organizzate per **rispondere a domande specifiche** che un ingegnere di pista si porrebbe:  

---

Per il Modello PIT (Quando pittare?)
| Categoria | Features | Domanda |
|-----------|----------|----------|
| Temporali | LapsRemainingNorm, LapsRemainingPct | "Quanto manca alla fine?" |
| Stato gomme | TyreMargin, TyreAgeCubeRatio, PitUrgency | "Le gomme possono reggere?" |
| Degrado misurato | DeltaFromBest, MaxSectorDeg, RearDeg | "Quanto sto perdendo?" |
| Trend | LapTimeTrend3, LapTimeTrend5, GapTrend | "Sto peggiorando?" |
| Confronto campo | MyStintVsField, MyTyreVsField | "Rispetto agli altri?" |
| Safety Car | UnderCaution | "In che posizione mi trovo?" |
| Telemetria | SpeedVsField, SpeedTrend3, SpeedLoss | "Quanta velocità perdo?" |

In [11]:
FEATURES_PIT = [
    'LapsRemainingNorm',
    'LapsRemainingPct',
    'TyreMargin',
    'DeltaFromBest',
    'MaxSectorDeg',
    'RearDeg',
    'SpeedLoss',
    'SpeedTrend3',
    'PitUrgency',
    'TyreAgeCubeRatio',
    'MyStintVsField',
    'MyTyreVsField',
    'GapTrend',
    'LapTimeTrend3',
    'LapTimeTrend5',
    'RaceProgress',
    'UnderCaution',
]

print(f"Features PIT: {len(FEATURES_PIT)}")

Features PIT: 17


Per il Modello COMPOUND (Quale gomma montare?)
| Categoria | Features | Domanda |
|-----------|----------|----------|
| Compound attuale | IsMedium, IsHard, IsSoft, IsInter | "Cosa ho adesso?" |
| Strategia | MustChangeCompound, HasUsedHard, NumCompoundsUsed | "Cosa devo fare per regolamento?" |
| Contesto | RaceProgress, LapsRemainingPct, StintNum, TyreMargin | "A che punto sono?" |
| Condizioni | IsRaining, TrackTempNorm | "Che tempo fa?" |
| Posizione | PosNorm, UnderCaution | "In che posizione mi trovo?" |
| Telemetria | SpeedLoss | "Quanta velocità perdo?" |

In [12]:
FEATURES_COMPOUND = [
    'IsSoft',
    'IsMedium',
    'IsHard',
    'IsInter',
    'NumCompoundsUsed',
    'HasUsedHard',
    'MustChangeCompound',
    'StintNum',
    'RaceProgress',
    'LapsRemainingPct',
    'IsRaining',
    'TrackTempNorm',
    'PosNorm',
    'TyreMargin',
    'UnderCaution',
    'SpeedLoss',
]

print(f"Features COMPOUND: {len(FEATURES_COMPOUND)}")

Features COMPOUND: 16


---
### 3.1 Features Temporali

*Perché sono importanti?*
Il timing del pit è fortemente influenzato dalla fase della gara:
- All'inizio della gara, i piloti tendono a spingere di più con gomme nuove e più morbide (Soft), rischiando di più, anche con gomme più degradabili. Questo porta a pit stop più frequenti.
- Verso la fine, ogni secondo perso ai box è molto costoso, quindi i piloti tendono a evitare pit stop se le gomme sono ancora in grado di mantenere un buon livello di performance. Strategia più conservativa per non perdere posizioni.

In [13]:
# Numero totale giri della gara (varia per circuito)
df_f1['TotalRaceLaps'] = df_f1.groupby(['Year', 'Round'])['LapNumber'].transform('max')

# Giri rimanenti - due normalizzazioni complementari:
df_f1['LapsRemaining'] = df_f1['TotalRaceLaps'] - df_f1['LapNumber']


# LapsRemainingPct: percentuale relativa alla gara specifica (0-1)
# Utile per confrontare circuiti con lunghezze diverse
df_f1['LapsRemainingPct'] = df_f1['LapsRemaining'] / df_f1['TotalRaceLaps']

print(df_f1['LapsRemainingPct'].describe())


count   86757.00
mean        0.49
std         0.28
min         0.00
25%         0.24
50%         0.49
75%         0.73
max         0.97
Name: LapsRemainingPct, dtype: float64


In [14]:
# LapsRemainingNorm: normalizzato su gara tipica (~70 giri)
# Utile per mantenere la scala assoluta dei giri
df_f1['LapsRemainingNorm'] = df_f1['LapsRemaining'] / 70

print(df_f1['LapsRemainingNorm'].describe())

count   86757.00
mean        0.43
std         0.26
min         0.00
25%         0.20
50%         0.41
75%         0.63
max         1.09
Name: LapsRemainingNorm, dtype: float64


---
### 3.2 Features Stato Gomme

*Perché sono importanti?*
Le gomme F1 hanno un "cliff": dopo un certo numero di giri il degrado accelera drasticamente a causa delle condizioni della pista.
- Un pilota può finire la gara con queste gomme? (Negativo = DEVE pittare) Solitamente 1 pit stop minimo è obbligatorio, e prevede un cambio di compound.
- Serve catturare il comportamento non lineare del degrado delle gomme nei giri avanzati.


In [15]:
# Valori basati su dati Pirelli e analisi telemetrie storiche
TYRE_CLIFF = {
    'SOFT': 18,        # Degrada velocemente, massime prestazioni iniziali
    'MEDIUM': 28,      # Bilanciata
    'HARD': 40,        # Lunga durata, meno grip iniziale
    'INTERMEDIATE': 35,
    'WET': 50          # Dipende molto dalla quantità di pioggia
}
df_f1['TyreCliff'] = df_f1['Compound'].map(TYRE_CLIFF).fillna(25)

In [16]:
# Età gomma rispetto al cliff (1.0 = siamo AL cliff)
df_f1['TyreAgeRatio'] = df_f1['TyreLife'] / df_f1['TyreCliff']

# Trasformazione cubica: amplifica i valori alti (vicini al cliff)
# Es: TyreAgeRatio=0.8 → TyreAgeCubeRatio=0.51 (poco effetto)
#     TyreAgeRatio=1.2 → TyreAgeCubeRatio=1.73 (molto effetto)
df_f1['TyreAgeCubeRatio'] = df_f1['TyreAgeRatio'] ** 3

print(df_f1['TyreAgeCubeRatio'].describe())

count   86574.00
mean        0.29
std         0.73
min         0.00
25%         0.02
50%         0.08
75%         0.28
max        20.80
Name: TyreAgeCubeRatio, dtype: float64


In [17]:
# Margine gomma: giri di vita rimanenti vs giri di gara rimanenti
# Positivo = può finire, Negativo = DEVE pittare
df_f1['TyreLifeRemaining'] = df_f1['TyreCliff'] - df_f1['TyreLife']
df_f1['TyreMargin'] = (df_f1['TyreLifeRemaining'] - df_f1['LapsRemaining']) / df_f1['TotalRaceLaps']

print(df_f1['TyreMargin'].describe())

count   86574.00
mean       -0.19
std         0.30
min        -0.82
25%        -0.45
50%        -0.16
75%         0.04
max         1.04
Name: TyreMargin, dtype: float64


In [18]:
# PitUrgency: combinazione di età gomma e fase gara
# Alta urgenza = gomme vecchie + pochi giri rimanenti
df_f1['PitUrgency'] = df_f1['TyreAgeRatio'] * (1 - df_f1['LapsRemainingPct'])

print(df_f1['PitUrgency'].describe())

count   86574.00
mean        0.29
std         0.29
min         0.00
25%         0.07
50%         0.19
75%         0.41
max         2.75
Name: PitUrgency, dtype: float64


---
### 3.3 Features Degrado

*Perché sono importanti?*
Le stime teoriche (cliff) non bastano: serve misurare il degrado REALE dai tempi.
- Quanto sto perdendo rispetto al mio miglior tempo della gara attuale?
- Quale settore soffre di più? (Può indicare problemi specifici)

In [19]:
def get_best_excluding_outlap(x):
    """
    Calcola il miglior tempo escludendo il primo giro (outlap) dopo il pit.
    L'outlap è sempre più lento: gomme fredde + uscita pit lane.
    """
    return x.iloc[1:].min() if len(x) > 1 else x.min()

In [20]:
stint_grp = df_f1.groupby(['Year', 'Round', 'Driver', 'Stint'])

# Delta dal miglior tempo dello stint (misura degrado complessivo)
df_f1['StintBestLap'] = stint_grp['LapTimeSec'].transform(get_best_excluding_outlap)
df_f1['DeltaFromBest'] = (df_f1['LapTimeSec'] - df_f1['StintBestLap']) / df_f1['StintBestLap']

print(df_f1['DeltaFromBest'].describe())

count   86757.00
mean        0.01
std         0.02
min        -0.15
25%         0.00
50%         0.01
75%         0.01
max         0.79
Name: DeltaFromBest, dtype: float64


In [21]:
# Degradazione per settore (identifica DOVE la gomma soffre)
for s in [1, 2, 3]:
    col = f'Sector{s}TimeSec'
    if col in df_f1.columns:
        best = stint_grp[col].transform(get_best_excluding_outlap)
        df_f1[f'S{s}Delta'] = (df_f1[col] - best) / best
    else:
        df_f1[f'S{s}Delta'] = 0
# Fill NaN nei delta settori
for col in ['S1Delta', 'S2Delta', 'S3Delta']:
    df_f1[col] = df_f1[col].fillna(0)

# RearDeg: degrado posteriore (S2+S3 sono più sensibili alle gomme posteriori)
# In F1 le gomme posteriori sono più sollecitate dalla trazione
df_f1['RearDeg'] = (df_f1['S2Delta'] + df_f1['S3Delta']) / 2

print(df_f1['RearDeg'].describe())

count   86757.00
mean        0.01
std         0.02
min        -0.15
25%         0.01
50%         0.01
75%         0.02
max         0.67
Name: RearDeg, dtype: float64


In [22]:
# MaxSectorDeg: settore con peggior degrado (identifica il punto critico)
df_f1['MaxSectorDeg'] = df_f1[['S1Delta', 'S2Delta', 'S3Delta']].max(axis=1)

print(df_f1['MaxSectorDeg'].describe())

count   86757.00
mean        0.02
std         0.05
min        -0.13
25%         0.01
50%         0.02
75%         0.03
max         2.90
Name: MaxSectorDeg, dtype: float64


---
### 3.4 Features Performance

*Perché sono importanti?*
Non basta sapere quanto sto perdendo ORA, serve sapere se sto PEGGIORANDO di giro in giro!
- Se il trend temporale delle performance del pilota mostra che i tempi per giro stanno peggiorando (ad esempio, il tempo di ogni giro aumenta), significa che le gomme stanno perdendo prestazione a causa del degrado.
- Outlap lento NON SIGNIFICA pit = gomma si sta scaldando -> può continuare

In [23]:
driver_grp = df_f1.groupby(['Year', 'Round', 'Driver'])

# Medie mobili tempi sul giro (per ridurre rumore e vedere meglio le tendenze generali)
df_f1['LapTimeMA3'] = driver_grp['LapTimeSec'].transform(lambda x: x.rolling(3, min_periods=1).mean())
# Media dei tempi sul giro calcolata su una finestra di 3 giri.

# Trend: deviazione dalla media mobile = quanto un dato tempo sul giro si discosta dalla sua media mobile.
#     Positivo = indica che il tempo sul giro sta peggiorando rispetto alla media recente, il che significa che il pilota sta rallentando.
#     Negativo = indica che il tempo sul giro sta migliorando rispetto alla media, il che potrebbe significare che le gomme si stanno scaldando o che le condizioni di gara stanno migliorando (ad esempio, meno traffico o cambiamento nelle condizioni meteo).
df_f1['LapTimeTrend3'] = (df_f1['LapTimeSec'] - df_f1['LapTimeMA3']) / df_f1['LapTimeMA3']

print(df_f1['LapTimeTrend3'].describe())

count   86757.00
mean       -0.00
std         0.01
min        -0.27
25%        -0.00
50%        -0.00
75%         0.00
max         0.41
Name: LapTimeTrend3, dtype: float64


In [24]:
df_f1['LapTimeMA5'] = driver_grp['LapTimeSec'].transform(lambda x: x.rolling(5, min_periods=1).mean())
# Media dei tempi sul giro calcolata su una finestra di 5 giri.

# Trend: deviazione dalla media mobile = quanto un dato tempo sul giro si discosta dalla sua media mobile.
#     Positivo = indica che il tempo sul giro sta peggiorando rispetto alla media recente, il che significa che il pilota sta rallentando.
#     Negativo = indica che il tempo sul giro sta migliorando rispetto alla media, il che potrebbe significare che le gomme si stanno scaldando o che le condizioni di gara stanno migliorando (ad esempio, meno traffico o cambiamento nelle condizioni meteo).
df_f1['LapTimeTrend5'] = (df_f1['LapTimeSec'] - df_f1['LapTimeMA5']) / df_f1['LapTimeMA5']

print(df_f1['LapTimeTrend5'].describe())

count   86757.00
mean       -0.00
std         0.01
min        -0.27
25%        -0.00
50%        -0.00
75%         0.00
max         0.54
Name: LapTimeTrend5, dtype: float64


---
### 3.5 Features Confronto

*Perché sono importanti?*
La strategia non dipende solo dal proprio stato, ma anche dal confronto con gli altri:
- Se ho gomme più vecchie degli altri devo anticipare il pit
- Se sto perdendo sul leader potrebbe convenire cambiare strategia

In [25]:
# Gap trend: sto guadagnando o perdendo sul leader?
df_f1['GapToLeader'] = df_f1['GapToLeader'].fillna(0)

# La colonna GapDiff contiene la variazione di tempo/distanza tra un giro e
# l'altro per ciascun pilota. Un valore positivo significa che il pilota
# sta perdendo terreno rispetto al leader, mentre un valore negativo indica che sta guadagnando.
df_f1['GapDiff'] = driver_grp['GapToLeader'].transform(lambda x: x.diff().fillna(0))
# Calcola una media mobile sui valori di GapDiff su una finestra di 3 giri.
df_f1['GapTrend'] = driver_grp['GapDiff'].transform(lambda x: x.rolling(3, min_periods=1).mean())

print(df_f1['GapTrend'].describe())

count   86757.00
mean        1.00
std         3.65
min       -48.28
25%         0.09
50%         0.97
75%         1.85
max        75.37
Name: GapTrend, dtype: float64


In [26]:
# Confronto con il campo sullo stesso giro
lap_grp = df_f1.groupby(['Year', 'Round', 'LapNumber'])
# Calcola l'età media delle gomme per tutti i piloti in ciascun giro della gara
df_f1['FieldAvgTyre'] = lap_grp['TyreLife'].transform('mean')
df_f1['MyTyreVsField'] = (df_f1['TyreLife'] - df_f1['FieldAvgTyre']) / (df_f1['FieldAvgTyre'] + 1)
# Un valore positivo indica che il pilota ha gomme più vecchie rispetto alla media, quindi potrebbe essere in svantaggio.
# Un valore negativo indica che il pilota ha gomme più fresche rispetto alla media, quindi potrebbe essere avvantaggiato.

print(df_f1['MyTyreVsField'].describe())

count   86574.00
mean        0.00
std         0.38
min        -0.95
25%        -0.20
50%         0.00
75%         0.17
max         4.46
Name: MyTyreVsField, dtype: float64


In [27]:
# Posizione normalizzata
df_f1['PosNorm'] = df_f1['Position'] / 20

print(df_f1['PosNorm'].describe())

count   86757.00
mean        0.48
std         0.27
min         0.05
25%         0.25
50%         0.50
75%         0.70
max         1.00
Name: PosNorm, dtype: float64


In [28]:
# Calcola la media del numero di stint (set di giri tra due pit stop) per tutti i piloti in ciascun giro.
df_f1['FieldAvgStint'] = lap_grp['Stint'].transform('mean')
# Un valore positivo indica che il pilota ha effettuato più stint degli altri, suggerendo una strategia più aggressiva (ad esempio, fermarsi meno o con giri più lunghi per stint).
# Un valore negativo indica che il pilota ha meno stint, suggerendo una strategia più conservativa.
df_f1['MyStintVsField'] = df_f1['Stint'] - df_f1['FieldAvgStint']

print(df_f1['MyStintVsField'].describe())

count   86757.00
mean        0.00
std         0.42
min        -2.12
25%        -0.17
50%        -0.05
75%         0.13
max         3.11
Name: MyStintVsField, dtype: float64


---
### 3.6 Features Condizioni

*Perché sono importanti?*
Per predire il COMPOUND, il modello deve sapere:
- Cosa ho ADESSO (one-hot encoding del compound corrente)
- Le condizioni meteo (determinano se servono gomme da bagnato)
- Temperatura pista (influenza degrado e aderenza)

In [29]:
# Temperatura pista normalizzata (centrata su 30°C, tipica temperatura di gara)
df_f1['TrackTempNorm'] = (df_f1['TrackTemp'] - 30) / 20
# Una temperatura della pista più alta o più bassa può influire sul degrado delle gomme e
# sulla strategia di pit stop, quindi è importante normalizzare per avere una scala coerente.

print(df_f1['TrackTempNorm'].describe())

count   86757.00
mean        0.30
std         0.44
min        -0.63
25%         0.03
50%         0.29
75%         0.66
max         1.22
Name: TrackTempNorm, dtype: float64


In [30]:
# Flag pioggia (binario: determina scelta INTER/WET)
df_f1['IsRaining'] = df_f1['Rainfall'].astype(int) if 'Rainfall' in df_f1.columns else 0

print(df_f1['IsRaining'].value_counts())

IsRaining
0    66806
1    19951
Name: count, dtype: int64


In [31]:
# Stint number normalizzato (clippato a 4: dopo il 4° stint la strategia è compromessa, > 4 = 4)
df_f1['StintNum'] = df_f1['Stint'].clip(upper=4) / 4

print(df_f1['StintNum'].describe())

count   86757.00
mean        0.51
std         0.22
min         0.25
25%         0.25
50%         0.50
75%         0.75
max         1.00
Name: StintNum, dtype: float64


In [32]:
# Sapere quale compound di gomme il pilota sta utilizzando in quel momento è
# cruciale per prevedere quale compound verrà montato nel prossimo pit stop.
#Ad esempio, se un pilota ha le gomme SOFT, probabilmente passerà a MEDIUM o
# HARD durante il prossimo pit stop, ma non cambierà a SOFT di nuovo.
df_f1['IsSoft'] = (df_f1['Compound'] == 'SOFT').astype(int)
df_f1['IsMedium'] = (df_f1['Compound'] == 'MEDIUM').astype(int)
df_f1['IsHard'] = (df_f1['Compound'] == 'HARD').astype(int)
df_f1['IsInter'] = (df_f1['Compound'] == 'INTERMEDIATE').astype(int)
# Nota: IsWet non incluso nelle features finali (troppo raro per essere predittivo)

print(df_f1[['IsSoft', 'IsMedium', 'IsHard', 'IsInter']].describe())

        IsSoft  IsMedium   IsHard  IsInter
count 86757.00  86757.00 86757.00 86757.00
mean      0.12      0.37     0.46     0.05
std       0.32      0.48     0.50     0.22
min       0.00      0.00     0.00     0.00
25%       0.00      0.00     0.00     0.00
50%       0.00      0.00     0.00     0.00
75%       0.00      1.00     1.00     0.00
max       1.00      1.00     1.00     1.00


---
### 3.7 Features Strategiche

*Perché sono importanti?*
Il regolamento F1 impone vincoli strategici:
- In gara asciutta, DEVONO essere usati almeno 2 compound diversi. Questo influenza fortemente la scelta del prossimo compound
- nel caso di incidente grave, interviene la Safety Car e il Pit Stop "costa la metà"!

In [33]:
def get_compounds_used(group):
    """
    Traccia quali compound sono già stati usati nella gara.
    Utile per capire se il pilota DEVE ancora cambiare compound per regolamento.
    """
    group = group.sort_values('LapNumber')
    used_hard = np.zeros(len(group))
    used_soft = np.zeros(len(group))
    used_medium = np.zeros(len(group))
    used_inter = np.zeros(len(group))
    used_wet = np.zeros(len(group))

    seen = set()
    current_stint = -1

    for i, (idx, row) in enumerate(group.iterrows()):
        if row['Stint'] != current_stint:
            seen.add(row['Compound'])
            current_stint = row['Stint']

        # Imposta il flag per ogni tipo di compound
        if 'HARD' in seen:
            used_hard[i] = 1
        if 'SOFT' in seen:
            used_soft[i] = 1
        if 'MEDIUM' in seen:
            used_medium[i] = 1
        if 'INTERMEDIATE' in seen:
            used_inter[i] = 1
        if 'WET' in seen:
            used_wet[i] = 1

    return pd.DataFrame({
        'HasUsedHard': used_hard,
        'HasUsedSoft': used_soft,
        'HasUsedMedium': used_medium,
        'HasUsedInter': used_inter,
        'HasUsedWet': used_wet
    }, index=group.index)

In [34]:
compounds_used = df_f1.groupby(['Year', 'Round', 'Driver']).apply(get_compounds_used)
compounds_used = compounds_used.reset_index(level=[0,1,2], drop=True)

df_f1[compounds_used.columns] = compounds_used

print(df_f1[['HasUsedHard', 'HasUsedSoft', 'HasUsedMedium', 'HasUsedInter']].describe())

       HasUsedHard  HasUsedSoft  HasUsedMedium  HasUsedInter
count     86757.00     86757.00       86757.00      86757.00
mean          0.56         0.24           0.74          0.09
std           0.50         0.43           0.44          0.28
min           0.00         0.00           0.00          0.00
25%           0.00         0.00           0.00          0.00
50%           1.00         0.00           1.00          0.00
75%           1.00         0.00           1.00          0.00
max           1.00         1.00           1.00          1.00


In [35]:
# Conta i compound usati
df_f1['NumCompoundsUsed'] = df_f1[['HasUsedHard', 'HasUsedSoft', 'HasUsedMedium', 'HasUsedInter']].sum(axis=1)

print(df_f1['NumCompoundsUsed'].describe())

count   86757.00
mean        1.64
std         0.57
min         0.00
25%         1.00
50%         2.00
75%         2.00
max         4.00
Name: NumCompoundsUsed, dtype: float64


In [36]:
# Flag che indica se il pilota deve ancora cambiare compound per regolamento
df_f1['MustChangeCompound'] = ((df_f1['NumCompoundsUsed'] < 2) & (df_f1['Stint'] >= 2)).astype(int)

print(df_f1['MustChangeCompound'].describe())

count   86757.00
mean        0.10
std         0.30
min         0.00
25%         0.00
50%         0.00
75%         0.00
max         1.00
Name: MustChangeCompound, dtype: float64


### 3.8 Features Telemetria
*Perché sono importanti?*
Le velocità nei punti di misura (speed traps, diversi settori) rivelano il **degrado reale** delle gomme:
- Calo di SpeedST -> perdita di grip -> pit imminente
- Queste features sono complementari a quelle basate sui tempi settore, essendo una misura più diretta delle prestazioni gomme

In [ ]:
stint_grp = df_f1.groupby(['Year', 'Round', 'Driver', 'Stint'])

def get_stint_max_speed(group):
    """Velocità massima nei primi 3 giri dello stint (gomme fresche)"""
    early_laps = group[group['TyreLife'] <= 3]['SpeedST']
    return early_laps.max() if len(early_laps) > 0 else group['SpeedST'].max()

stint_max_speed = stint_grp.apply(get_stint_max_speed).rename('StintMaxSpeed')
df_f1 = df_f1.merge(
    stint_max_speed.reset_index(),
    on=['Year', 'Round', 'Driver', 'Stint'],
    how='left'
)

In [ ]:
# Perdita velocità rispetto a inizio stint (negativa = sto perdendo)
df_f1['SpeedLoss'] = (df_f1['SpeedST'] - df_f1['StintMaxSpeed']) / df_f1['StintMaxSpeed']
df_f1['SpeedLoss'] = df_f1['SpeedLoss'].fillna(0).clip(-0.2, 0.05)  # Clip outliers

print(df_f1['SpeedLoss'].describe())

In [ ]:
# Trend velocità ultimi 3 giri
driver_grp = df_f1.groupby(['Year', 'Round', 'Driver'])
df_f1['SpeedTrend3'] = driver_grp['SpeedST'].transform(
    lambda x: x.diff().rolling(3, min_periods=1).mean()
)
df_f1['SpeedTrend3'] = df_f1['SpeedTrend3'].fillna(0).clip(-5, 5)  # km/h per giro

print(df_f1['SpeedTrend3'].describe())

---
# 4. Preparazione Dati per Training

### 4.1 Pulizia e Validazione

In [ ]:
# Gestione valori infiniti (da divisioni nelle features calcolate)
df_f1 = df_f1.replace([np.inf, -np.inf], np.nan)

# Fill NaN nelle features con mediana
all_features = list(set(FEATURES_PIT + FEATURES_COMPOUND))
for f in all_features:
    if f in df_f1.columns:
        df_f1[f] = df_f1[f].fillna(df_f1[f].median())

In [ ]:
# Filtraggio per training
df_clean = df_f1.dropna(subset=['PitIn3Laps', 'NextCompound'])  # Target validi
df_clean = df_clean[df_clean['TyreLife'] > 1]                   # Escludi outlap


# Encoding target compound
label_encoder = LabelEncoder()
df_clean['NextCompoundEnc'] = label_encoder.fit_transform(df_clean['NextCompound'])
print(f"\nClassi: {list(label_encoder.classes_)}")


Classi: ['HARD', 'INTERMEDIATE', 'MEDIUM', 'SOFT', 'WET']


### 4.2 Creazione Sequenze per LSTM

*Perché sequenze di 10 giri?*
L'LSTM ha bisogno di contesto temporale per catturare i trend: 10 giri è un buon compromesso: abbastanza per vedere il degrado, non troppo per includere rumore. Stint più corti vengono paddati con zeri (il Masking layer li ignorerà)

In [ ]:
SEQUENCE_LENGTH = 10  # Giri di storia per la predizione

# Crea sequenze temporali per LSTM
def create_sequences(df, features, target_col, seq_len):
    """
    Crea sequenze temporali per LSTM.
    Input: (samples, timesteps, features) = (N, 10, 14)
    Per ogni giro, prende i 'seq_len' giri precedenti come input.
    """
    X = []  # Input = sequenza di giri
    y = []  # Output = target (PitIn3Laps o NextCompoundEnc)
    available = [f for f in features if f in df.columns] # Filtra features esistenti

    for (year, rnd, driver, stint), group in df.groupby(['Year', 'Round', 'Driver', 'Stint']):
        group = group.sort_values('LapNumber')
        if len(group) < 2:  # Skip stint troppo corti
            continue

        data = group[available].values.astype(np.float32)
        targets = group[target_col].values

        # Per ogni giro, prendi i 'seq_len' giri precedenti come input per creare la sequenza
        for i in range(1, len(group)):  # Inizia da 1 (serve almeno 1 giro di storia)
            start = max(0, i - seq_len)
            seq = data[start:i]

            # Padding se sequenza troppo corta (padding con zeri)
            if len(seq) < seq_len:
                pad = np.zeros((seq_len - len(seq), len(available)), dtype=np.float32)
                seq = np.vstack([pad, seq])

            X.append(seq)
            y.append(targets[i])

    return np.array(X), np.array(y), available

In [ ]:
# Filtra features esistenti
FEATURES_PIT = [f for f in FEATURES_PIT if f in df_clean.columns]
FEATURES_COMPOUND = [f for f in FEATURES_COMPOUND if f in df_clean.columns]
print(f"Features PIT effettive: {len(FEATURES_PIT)}")
print(f"Features COMPOUND effettive: {len(FEATURES_COMPOUND)}")

print("\nCreazione sequenze PIT...")
X_pit, y_pit, feat_pit = create_sequences(df_clean, FEATURES_PIT, 'PitIn3Laps', SEQUENCE_LENGTH)
print(f"  Shape: {X_pit.shape}")
print(f"  Classe positiva: {y_pit.mean():.1%}") # ~80% dei giri NON sono seguiti da pit

print("\nCreazione sequenze COMPOUND...")
X_comp, y_comp, feat_comp = create_sequences(df_clean, FEATURES_COMPOUND, 'NextCompoundEnc', SEQUENCE_LENGTH)
print(f"  Shape: {X_comp.shape}")

Features PIT effettive: 16
Features COMPOUND effettive: 15

Creazione sequenze PIT...
  Shape: (81979, 10, 16)
  Classe positiva: 21.5%

Creazione sequenze COMPOUND...
  Shape: (81979, 10, 15)


### 4.3 Split Train/Validation/Test

**Split Strategy:**
- Train: 70% - per apprendimento
- Validation: 15% - per early stopping e tuning
- Test: 15% - per valutazione finale

**Nota di miglioramento:** Usiamo stratified split (dividendo i dati con la stessa proporzione delle classi del dataset originale.) invece di time-based per semplicità.

In [ ]:
# Split PIT (stratificato per mantenere proporzione pit/no-pit)
# Uso la stratificazione perché il target è sbilanciato:
# senza stratify il modello vedrebbe set con percentuali pit/no-pit diverse,
# falsando validazione e test e rendendo le metriche non confrontabili.
X_pit_temp, X_pit_test, y_pit_temp, y_pit_test = train_test_split(
    X_pit, y_pit, test_size=0.15, random_state=42, stratify=y_pit
)
X_pit_train, X_pit_val, y_pit_train, y_pit_val = train_test_split(
    X_pit_temp, y_pit_temp, test_size=0.176, random_state=42, stratify=y_pit_temp
)

# Split COMPOUND (stratificato per mantenere proporzione delle mescole)
# Necessario perché alcune mescole sono molto meno frequenti:
# la stratificazione evita che val/test manchino di classi rare.
X_comp_temp, X_comp_test, y_comp_temp, y_comp_test = train_test_split(
    X_comp, y_comp, test_size=0.15, random_state=42, stratify=y_comp
)
X_comp_train, X_comp_val, y_comp_train, y_comp_val = train_test_split(
    X_comp_temp, y_comp_temp, test_size=0.176, random_state=42, stratify=y_comp_temp
)

print("Split completato:")
print(f"  PIT   - Train: {len(X_pit_train):,} | Val: {len(X_pit_val):,} | Test: {len(X_pit_test):,}")
print(f"  COMP  - Train: {len(X_comp_train):,} | Val: {len(X_comp_val):,} | Test: {len(X_comp_test):,}")

Split completato:
  PIT   - Train: 57,417 | Val: 12,265 | Test: 12,297
  COMP  - Train: 57,417 | Val: 12,265 | Test: 12,297


### 4.4 Normalizzazione e Class Weights

**Normalizzazione**: RobustScaler usa mediana invece di media
- *Perché?* Più robusto agli outliers (pit lap, safety car, outlap)

**Class Weights**: Bilancia le classi sbilanciate
- *Perché?* ~80% dei giri NON sono seguiti da pit -> il modello tenderebbe a predire sempre "no pit"

In [ ]:
# Normalizzazione PIT
scaler_pit = RobustScaler()
n_features_pit = X_pit_train.shape[2] #   Shape: (81979, 10, 16)

X_pit_train = scaler_pit.fit_transform(X_pit_train.reshape(-1, n_features_pit)).reshape(X_pit_train.shape)
# = Shape: (81979, 10, 16) -> (819790, 16) -> norm -> (81979, 10, 16)
X_pit_val = scaler_pit.transform(X_pit_val.reshape(-1, n_features_pit)).reshape(X_pit_val.shape)
X_pit_test = scaler_pit.transform(X_pit_test.reshape(-1, n_features_pit)).reshape(X_pit_test.shape)


# Normalizzazione COMPOUND
scaler_comp = RobustScaler()
n_features_comp = X_comp_train.shape[2]   # Shape: (81979, 10, 15)

X_comp_train = scaler_comp.fit_transform(X_comp_train.reshape(-1, n_features_comp)).reshape(X_comp_train.shape)
# = Shape: (81979, 10, 15) -> (819790, 15) -> norm -> (81979, 10, 15)
X_comp_val = scaler_comp.transform(X_comp_val.reshape(-1, n_features_comp)).reshape(X_comp_val.shape)
X_comp_test = scaler_comp.transform(X_comp_test.reshape(-1, n_features_comp)).reshape(X_comp_test.shape)

In [ ]:
# Class weights: weight = n_samples / (n_classes * n_samples_per_class)
# Classe rara -> weight alto

pit_counts = np.bincount(y_pit_train.astype(int))
pit_weight = {
    0: 1.0, # Classe di riferimento
    1: pit_counts[0] / pit_counts[1] # Classe rara
}
print(f"\nPit class weights:")
print(f"  Classe 0 (No Pit): {pit_weight[0]:.2f}")
print(f"  Classe 1 (Pit):    {pit_weight[1]:.2f}")

comp_counts = np.bincount(y_comp_train)
n_samples = len(y_comp_train)
comp_weight = { # non esiste una classe di riferimento
    0: n_samples / (5 * comp_counts[0]),
    1: n_samples / (5 * comp_counts[1]),
    2: n_samples / (5 * comp_counts[2]),
    3: n_samples / (5 * comp_counts[3]),
    4: n_samples / (5 * comp_counts[4]),
}
print(f"\nCompound class weights:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {cls:12s}: {comp_weight[i]:.2f} (n={comp_counts[i]:,})")

Normalizzazione completata!

Pit class weights:
  Classe 0 (No Pit): 1.00
  Classe 1 (Pit):    3.64

Compound class weights:
  HARD        : 0.38 (n=29,961)
  INTERMEDIATE: 4.44 (n=2,586)
  MEDIUM      : 0.71 (n=16,134)
  SOFT        : 1.33 (n=8,649)
  WET         : 131.99 (n=87)
